In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Import required libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os
import time

# Step 3: Set parameters
IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 1
BATCH_SIZE = 32
LATENT_DIM = 100
EPOCHS = 100  # You can increase this

# Path configurations
data_dir = "/content/drive/MyDrive/chest_xray/train/PNEUMONIA"
save_dir = "/content/drive/MyDrive/GAN_Models"
output_dir = "/content/drive/MyDrive/Generated_Images"

# Create directories if they don't exist
os.makedirs(save_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Step 4: Create dataset
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    directory=data_dir,
    label_mode=None,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    shuffle=True
)

# Normalize images to [-1, 1]
def preprocess(image):
    return (image / 127.5) - 1.0

train_dataset = train_dataset.map(lambda x: preprocess(x))

# Step 5: Build Generator
def build_generator():
    model = keras.Sequential()
    # Foundation for 8x8 image
    model.add(layers.Dense(8*8*256, use_bias=False, input_dim=LATENT_DIM))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((8, 8, 256)))

    # Upsample to 16x16
    model.add(layers.Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Upsample to 32x32
    model.add(layers.Conv2DTranspose(64, (4,4), strides=(2,2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Upsample to 64x64
    model.add(layers.Conv2DTranspose(32, (4,4), strides=(2,2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Upsample to 128x128
    model.add(layers.Conv2DTranspose(1, (4,4), strides=(2,2), padding='same', use_bias=False, activation='tanh'))

    return model

# Step 6: Build Discriminator
def build_discriminator():
    model = keras.Sequential()
    model.add(layers.Conv2D(32, (5,5), strides=(2,2), padding='same',
                                     input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(64, (5,5), strides=(2,2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5,5), strides=(2,2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

# Step 7: Build and compile GAN
generator = build_generator()
discriminator = build_discriminator()

discriminator.compile(loss=keras.losses.BinaryCrossentropy(from_logits=True),
                      optimizer=keras.optimizers.Adam(1e-4),
                      metrics=['accuracy'])

discriminator.trainable = False
gan_input = layers.Input(shape=(LATENT_DIM,))
gan_output = discriminator(generator(gan_input))
gan = keras.Model(gan_input, gan_output)

gan.compile(loss=keras.losses.BinaryCrossentropy(from_logits=True),
            optimizer=keras.optimizers.Adam(1e-4))

# Step 8: Training functions
def generate_and_save_images(model, epoch, examples=3):
    noise = tf.random.normal([examples, LATENT_DIM])
    generated_images = model(noise, training=False)
    generated_images = (generated_images + 1) * 127.5  # Rescale to 0-255

    plt.figure(figsize=(10, 10))
    for i in range(examples):
        plt.subplot(1, examples, i+1)
        plt.imshow(generated_images[i].numpy().reshape(IMG_HEIGHT, IMG_WIDTH), cmap='gray')
        plt.axis('off')
    plt.savefig(f"{output_dir}/epoch_{epoch+1}.png")
    plt.close()

def train_gan(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()

        for real_images in dataset:
            # Train Discriminator
            noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])
            generated_images = generator(noise, training=False)

            real_labels = tf.ones((BATCH_SIZE, 1)) * 0.9  # Label smoothing
            fake_labels = tf.zeros((BATCH_SIZE, 1))

            # Train discriminator on real and fake images
            d_loss_real = discriminator.train_on_batch(real_images, real_labels)
            d_loss_fake = discriminator.train_on_batch(generated_images, fake_labels)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train Generator
            noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])
            g_loss = gan.train_on_batch(noise, tf.ones((BATCH_SIZE, 1)))

        # Save models and generate images every 10 epochs
        if (epoch + 1) % 10 == 0:
            generator.save(f"{save_dir}/generator_epoch_{epoch+1}.h5")
            discriminator.save(f"{save_dir}/discriminator_epoch_{epoch+1}.h5")
            generate_and_save_images(generator, epoch)

        print(f"Epoch {epoch+1}, Time: {time.time()-start:.2f}s")
        print(f"D Loss: {d_loss[0]:.4f}, D Acc: {d_loss[1]*100:.2f}%")
        print(f"G Loss: {g_loss:.4f}\n")

# Step 9: Start training
train_gan(train_dataset, EPOCHS)

Mounted at /content/drive
Found 1857 files.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:82: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Epoch 1, Time: 83.33s
D Loss: 0.7044, D Acc: 49.79%
G Loss: 0.5146

Epoch 2, Time: 10.44s
D Loss: 0.7121, D Acc: 49.89%
G Loss: 0.4451

Epoch 3, Time: 9.01s
D Loss: 0.7255, D Acc: 49.93%
G Loss: 0.3962

Epoch 4, Time: 11.06s
D Loss: 0.7480, D Acc: 47.74%
G Loss: 0.3579

Epoch 5, Time: 8.43s
D Loss: 0.7792, D Acc: 38.57%
G Loss: 0.3273

Epoch 6, Time: 20.49s
D Loss: 0.8170, D Acc: 32.14%
G Loss: 0.3022

Epoch 7, Time: 9.04s
D Loss: 0.8619, D Acc: 27.54%
G Loss: 0.2805

Epoch 8, Time: 10.24s
D Loss: 0.9127, D Acc: 24.10%
G Loss: 0.2614

Epoch 9, Time: 10.24s
D Loss: 0.9676, D Acc: 21.42%
G Loss: 0.2443



Epoch 10, Time: 10.28s
D Loss: 1.0237, D Acc: 19.28%
G Loss: 0.2289

Epoch 11, Time: 8.77s
D Loss: 1.0794, D Acc: 17.52%
G Loss: 0.2150

Epoch 12, Time: 9.27s
D Loss: 1.1339, D Acc: 16.06%
G Loss: 0.2025

Epoch 13, Time: 9.82s
D Loss: 1.1865, D Acc: 14.83%
G Loss: 0.1912

Epoch 14, Time: 10.24s
D Loss: 1.2371, D Acc: 13.77%
G Loss: 0.1809

Epoch 15, Time: 10.23s
D Loss: 1.2860, D Acc: 12.85%
G Loss: 0.1717

Epoch 16, Time: 9.06s
D Loss: 1.3332, D Acc: 12.05%
G Loss: 0.1632

Epoch 17, Time: 10.23s
D Loss: 1.3783, D Acc: 11.34%
G Loss: 0.1555

Epoch 18, Time: 9.50s
D Loss: 1.4219, D Acc: 10.71%
G Loss: 0.1485

Epoch 19, Time: 8.27s
D Loss: 1.4638, D Acc: 10.14%
G Loss: 0.1420



Epoch 20, Time: 10.51s
D Loss: 1.5042, D Acc: 9.64%
G Loss: 0.1361

Epoch 21, Time: 9.58s
D Loss: 1.5432, D Acc: 9.18%
G Loss: 0.1306

Epoch 22, Time: 10.23s
D Loss: 1.5810, D Acc: 8.76%
G Loss: 0.1255

Epoch 23, Time: 10.24s
D Loss: 1.6175, D Acc: 8.38%
G Loss: 0.1208

Epoch 24, Time: 9.44s
D Loss: 1.6527, D Acc: 8.03%
G Loss: 0.1164

Epoch 25, Time: 10.23s
D Loss: 1.6868, D Acc: 7.71%
G Loss: 0.1123

Epoch 26, Time: 10.24s
D Loss: 1.7198, D Acc: 7.41%
G Loss: 0.1085

Epoch 27, Time: 8.16s
D Loss: 1.7519, D Acc: 7.14%
G Loss: 0.1049

Epoch 28, Time: 10.23s
D Loss: 1.7831, D Acc: 6.88%
G Loss: 0.1016

Epoch 29, Time: 9.50s
D Loss: 1.8132, D Acc: 6.65%
G Loss: 0.0985



...

Epoch 60, Time: 10.60s
D Loss: 2.4611, D Acc: 3.21%
G Loss: 0.0502

Epoch 61, Time: 10.24s
D Loss: 2.4761, D Acc: 3.16%
G Loss: 0.0494

Epoch 62, Time: 10.23s
D Loss: 2.4908, D Acc: 3.11%
G Loss: 0.0487

Epoch 63, Time: 10.24s
D Loss: 2.5052, D Acc: 3.06%
G Loss: 0.0480

Epoch 64, Time: 10.23s
D Loss: 2.5195, D Acc: 3.01%
G Loss: 0.0472

Epoch 65, Time: 10.23s
D Loss: 2.5335, D Acc: 2.96%
G Loss: 0.0466

Epoch 66, Time: 9.16s
D Loss: 2.5473, D Acc: 2.92%
G Loss: 0.0459

Epoch 67, Time: 10.23s
D Loss: 2.5609, D Acc: 2.88%
G Loss: 0.0452

Epoch 68, Time: 9.45s
D Loss: 2.5743, D Acc: 2.83%
G Loss: 0.0446

Epoch 69, Time: 8.34s
D Loss: 2.5875, D Acc: 2.79%
G Loss: 0.0440



Epoch 70, Time: 9.75s
D Loss: 2.6005, D Acc: 2.75%
G Loss: 0.0434

Epoch 71, Time: 9.50s
D Loss: 2.6133, D Acc: 2.71%
G Loss: 0.0428

Epoch 72, Time: 8.16s
D Loss: 2.6259, D Acc: 2.68%
G Loss: 0.0422

Epoch 73, Time: 9.65s
D Loss: 2.6383, D Acc: 2.64%
G Loss: 0.0417

Epoch 74, Time: 9.68s
D Loss: 2.6505, D Acc: 2.60%
G Loss: 0.0411

Epoch 75, Time: 8.16s
D Loss: 2.6626, D Acc: 2.57%
G Loss: 0.0406

Epoch 76, Time: 10.24s
D Loss: 2.6745, D Acc: 2.54%
G Loss: 0.0401

Epoch 77, Time: 10.23s
D Loss: 2.6863, D Acc: 2.50%
G Loss: 0.0396

Epoch 78, Time: 8.96s
D Loss: 2.6979, D Acc: 2.47%
G Loss: 0.0391

Epoch 79, Time: 10.24s
D Loss: 2.7093, D Acc: 2.44%
G Loss: 0.0386



Epoch 80, Time: 9.90s
D Loss: 2.7206, D Acc: 2.41%
G Loss: 0.0382

Epoch 81, Time: 9.87s
D Loss: 2.7317, D Acc: 2.38%
G Loss: 0.0377

Epoch 82, Time: 8.29s
D Loss: 2.7427, D Acc: 2.35%
G Loss: 0.0373

Epoch 83, Time: 9.56s
D Loss: 2.7536, D Acc: 2.32%
G Loss: 0.0369

Epoch 84, Time: 9.59s
D Loss: 2.7643, D Acc: 2.29%
G Loss: 0.0364

Epoch 85, Time: 8.52s
D Loss: 2.7749, D Acc: 2.27%
G Loss: 0.0360

Epoch 86, Time: 9.29s
D Loss: 2.7853, D Acc: 2.24%
G Loss: 0.0356

Epoch 87, Time: 9.67s
D Loss: 2.7956, D Acc: 2.21%
G Loss: 0.0352

Epoch 88, Time: 10.23s
D Loss: 2.8058, D Acc: 2.19%
G Loss: 0.0348

Epoch 89, Time: 10.23s
D Loss: 2.8159, D Acc: 2.17%
G Loss: 0.0345



Epoch 90, Time: 10.52s
D Loss: 2.8258, D Acc: 2.14%
G Loss: 0.0341

Epoch 91, Time: 10.24s
D Loss: 2.8357, D Acc: 2.12%
G Loss: 0.0337

Epoch 92, Time: 10.23s
D Loss: 2.8454, D Acc: 2.09%
G Loss: 0.0334

Epoch 93, Time: 8.41s
D Loss: 2.8550, D Acc: 2.07%
G Loss: 0.0330

Epoch 94, Time: 10.24s
D Loss: 2.8645, D Acc: 2.05%
G Loss: 0.0327

Epoch 95, Time: 9.48s
D Loss: 2.8738, D Acc: 2.03%
G Loss: 0.0324

Epoch 96, Time: 9.16s
D Loss: 2.8831, D Acc: 2.01%
G Loss: 0.0320

Epoch 97, Time: 10.25s
D Loss: 2.8923, D Acc: 1.99%
G Loss: 0.0317

Epoch 98, Time: 10.23s
D Loss: 2.9013, D Acc: 1.97%
G Loss: 0.0314

Epoch 99, Time: 10.23s
D Loss: 2.9103, D Acc: 1.95%
G Loss: 0.0311



Epoch 100, Time: 10.53s
D Loss: 2.9191, D Acc: 1.93%
G Loss: 0.0308



In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import os
import time
import imageio
import matplotlib.pyplot as plt
import numpy as np
from IPython import display
from tensorflow.keras import layers

# Set parameters
IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 1
BATCH_SIZE = 32
LATENT_DIM = 256
EPOCHS = 500
BUFFER_SIZE = 1000

# Path configurations
drive_path = "/content/drive/MyDrive"
data_dir = os.path.join(drive_path, "chest_xray/train/PNEUMONIA")
save_dir = os.path.join(drive_path, "GAN_Models")
output_dir = os.path.join(drive_path, "Generated_Xrays")

# Create directories
os.makedirs(save_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Load and preprocess dataset
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    directory=os.path.dirname(data_dir),  # Parent directory containing NORMAL/PNEUMONIA
    label_mode=None,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    shuffle=True
).map(lambda x: (x / 127.5) - 1.0)  # Normalize to [-1, 1]

# Improved Generator for 128x128 images
def make_generator():
    model = tf.keras.Sequential()
    model.add(layers.Dense(8*8*512, use_bias=False, input_shape=(LATENT_DIM,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    model.add(layers.Reshape((8, 8, 512)))

    # Upsample to 16x16
    model.add(layers.Conv2DTranspose(256, (5,5), strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Upsample to 32x32
    model.add(layers.Conv2DTranspose(128, (5,5), strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Upsample to 64x64
    model.add(layers.Conv2DTranspose(64, (5,5), strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Upsample to 128x128
    model.add(layers.Conv2DTranspose(32, (5,5), strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.2))

    # Final layer
    model.add(layers.Conv2DTranspose(1, (5,5), strides=1, padding='same', use_bias=False, activation='tanh'))

    return model

# Improved Discriminator
def make_discriminator():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5,5), strides=2, padding='same',
                                     input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5,5), strides=2, padding='same'))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(256, (5,5), strides=2, padding='same'))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

# Initialize models
generator = make_generator()
discriminator = make_discriminator()

# Loss functions and optimizers
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-5, beta_1=0.5)

# Checkpoint setup
checkpoint_prefix = os.path.join(save_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

# Training parameters
num_examples_to_generate = 9
seed = tf.random.normal([num_examples_to_generate, LATENT_DIM])

# Training functions
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(9, 9))

    for i in range(predictions.shape[0]):
        plt.subplot(3, 3, i+1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')

    plt.savefig(os.path.join(output_dir, f'xray_epoch_{epoch:04d}.png'))
    plt.close()

def train(dataset, epochs):
    # Try to restore latest checkpoint
    latest = tf.train.latest_checkpoint(save_dir)
    if latest:
        checkpoint.restore(latest)
        start_epoch = int(latest.split('_')[-1].split('.')[0])
        print(f"Resuming from epoch {start_epoch}")
    else:
        start_epoch = 0

    for epoch in range(start_epoch, epochs):
        start = time.time()

        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)

        # Generate and save images every epoch
        display.clear_output(wait=True)
        generate_and_save_images(generator, epoch + 1, seed)

        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
            print(f'Saved checkpoint at epoch {epoch+1}')

        print(f'Epoch {epoch+1} completed in {time.time()-start:.2f}s')
        print(f'Generator loss: {gen_loss:.4f}, Discriminator loss: {disc_loss:.4f}\n')

    # Generate final images
    display.clear_output(wait=True)
    generate_and_save_images(generator, epochs, seed)

# Start training
train(train_dataset, EPOCHS)

# Create GIF
anim_file = os.path.join(drive_path, 'xray_generation.gif')

with imageio.get_writer(anim_file, mode='I') as writer:
    filenames = sorted(glob.glob(os.path.join(output_dir, 'xray_epoch_*.png')))
    last_image = None
    for filename in filenames:
        image = imageio.imread(filename)
        if image != last_image:  # Skip duplicate frames
            writer.append_data(image)
        last_image = image
    # Add last frame 10 times to pause at end
    for _ in range(10):
        writer.append_data(image)

display.display(display.HTML(f'<img src="{anim_file}">'))

Epoch 468 completed in 25.57s
Generator loss: 0.7676, Discriminator loss: 1.2842

